# Загрузка и подготовка данных dataset_rework

**Шаг 1 плана.** Загрузка dataset_rework, подготовка сессий, переименование signal_barrier → rd_regime, добавление rd_regime_transition и фичей. Сохранение для следующих ноутбуков.

## 1. Импорты и настройка путей

In [1]:
import sys
import os
import numpy as np
import pandas as pd

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.data.dataset_rework_loader import load_dataset_rework, prepare_for_training
from src.features.feature_pipeline import add_features, get_feature_columns
from src.features.warmup_loader import add_warmup_from_bybit, remove_warmup

## 2. Загрузка dataset_rework и подготовка сессий

In [2]:
data_dir = os.path.join(_root, 'dataset_rework')
df_raw = load_dataset_rework(data_dir=data_dir, verbose=True)
df = prepare_for_training(df=df_raw, verbose=True)
df.head()

Найдено CSV: 833 | успешно прочитано: 833 | пропущено: 0
Загружено: 395,585 строк, 223 символов


Подготовлено: 365,375 строк, 1,048 сессий, 214 символов


,timestamp,symbol,rd_value,open,high,low,close_price,volume,signal_barrier,source_file,source_day,datetime,time_diff_min,session_key
98,1770320820000,4,-0.078203,0.009382,0.009382,0.009363,0.009366,250233.0,-1,4.csv,2026-02-06,2026-02-05 19:47:00+00:00,NaN,4_s11
99,1770320880000,4,-0.069046,0.009366,0.009383,0.009363,0.009367,29861.0,1,4.csv,2026-02-06,2026-02-05 19:48:00+00:00,1.0,4_s11
100,1770320940000,4,-0.044992,0.009367,0.009382,0.009367,0.009376,61323.0,1,4.csv,2026-02-06,2026-02-05 19:49:00+00:00,1.0,4_s11
101,1770321000000,4,-0.019496,0.009376,0.009389,0.009376,0.009389,7999.0,1,4.csv,2026-02-06,2026-02-05 19:50:00+00:00,1.0,4_s11
102,1770321060000,4,-0.092795,0.009389,0.009389,0.009330,0.009340,146393.0,-1,4.csv,2026-02-06,2026-02-05 19:51:00+00:00,1.0,4_s11


## 3. Переименование signal_barrier → rd_regime, rd_regime_transition

In [3]:
# Если признаки уже есть — не пересчитываем. Иначе строим из signal_barrier/rd_value.
if 'rd_regime' not in df.columns:
    if 'signal_barrier' in df.columns:
        df['rd_regime'] = pd.to_numeric(df['signal_barrier'], errors='coerce').fillna(0)
    else:
        df = df.sort_values(['session_key', 'datetime']).reset_index(drop=True)
        df['rd_regime'] = df.groupby('session_key')['rd_value'].diff().pipe(np.sign).fillna(0)

df['rd_regime'] = np.sign(pd.to_numeric(df['rd_regime'], errors='coerce').fillna(0)).astype(int)

if 'rd_regime_transition' not in df.columns:
    df['rd_regime_prev'] = df.groupby('session_key')['rd_regime'].shift(1)
    df['rd_regime_transition'] = ((df['rd_regime'] != df['rd_regime_prev']) & df['rd_regime_prev'].notna()).astype(int)
    df = df.drop(columns=['rd_regime_prev'], errors='ignore')
else:
    df['rd_regime_transition'] = pd.to_numeric(df['rd_regime_transition'], errors='coerce').fillna(0).astype(int)

print('rd_regime:', df['rd_regime'].value_counts().to_dict())
print('rd_regime_transition (доля переходов):', f"{df['rd_regime_transition'].mean():.4f}")

rd_regime: {-1: 183504, 1: 181871}
rd_regime_transition (доля переходов): 0.3808


## 4. Добавление фичей (с warmup Bybit перед расчётом)

In [4]:
USE_WARMUP = True
WARMUP_SIZE = 60

if USE_WARMUP:
    try:
        # Добавляем 60 минут warmup по каждой сессии, считаем фичи, затем удаляем warmup-строки.
        df_with_warmup = add_warmup_from_bybit(df, warmup_size=WARMUP_SIZE, verbose=True)
        df_feat, enc = add_features(df_with_warmup)
        df = remove_warmup(df_feat)
        print(f'Warmup применён: size={WARMUP_SIZE}. После удаления warmup строк: {len(df):,}')
    except Exception as e:
        print(f'Warmup пропущен ({e}). Считаем фичи без warmup.')
        df, enc = add_features(df)
else:
    df, enc = add_features(df)

feat_cols = get_feature_columns(include_symbol=True)
print(f'Фичей: {len(feat_cols)}')
df[feat_cols[:5] + ['rd_regime', 'rd_regime_transition']].head()

Warmup skip 4_s11: pybit не установлен. pip install pybit
Warmup skip 4_s13: pybit не установлен. pip install pybit
Warmup skip 4_s19: pybit не установлен. pip install pybit
Warmup skip 10000ELON_s0: pybit не установлен. pip install pybit


Warmup skip 10000QUBIC_s0: pybit не установлен. pip install pybit
Warmup skip 10000QUBIC_s3: pybit не установлен. pip install pybit
Warmup skip 10000QUBIC_s8: pybit не установлен. pip install pybit
Warmup skip 10000QUBIC_s9: pybit не установлен. pip install pybit
Warmup skip 10000QUBIC_s10: pybit не установлен. pip install pybit
Warmup skip 10000QUBIC_s12: pybit не установлен. pip install pybit
Warmup skip 1000BTT_s0: pybit не установлен. pip install pybit


Warmup skip 1000LUNC_s0: pybit не установлен. pip install pybit
Warmup skip 1000RATS_s0: pybit не установлен. pip install pybit
Warmup skip 1000RATS_s2: pybit не установлен. pip install pybit


Warmup skip 1000TAG_s0: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s1: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s2: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s5: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s12: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s13: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s14: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s16: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s17: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s19: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s21: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s23: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s24: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s26: pybit не установлен. pip install pybit
Warmup skip 1000TAG_s27: pybit не установлен. pip install pybit
Warmup skip 1000X_s1: pybit не установлен. p

Warmup skip 1INCH_s0: pybit не установлен. pip install pybit
Warmup skip ACT_s0: pybit не установлен. pip install pybit
Warmup skip ACT_s1: pybit не установлен. pip install pybit


...

Warmup skip ZIL_s1: pybit не установлен. pip install pybit
Warmup skip ZIL_s10: pybit не установлен. pip install pybit
Warmup skip ZIL_s14: pybit не установлен. pip install pybit
Warmup skip ZIL_s15: pybit не установлен. pip install pybit
Warmup skip ZIL_s25: pybit не установлен. pip install pybit
Warmup skip ZIL_s26: pybit не установлен. pip install pybit
Warmup skip ZIL_s33: pybit не установлен. pip install pybit
Warmup skip ZIL_s34: pybit не установлен. pip install pybit
Warmup skip ZIL_s38: pybit не установлен. pip install pybit
Warmup skip ZIL_s40: pybit не установлен. pip install pybit
Warmup skip ZIL_s43: pybit не установлен. pip install pybit
Warmup skip ZIL_s45: pybit не установлен. pip install pybit
Warmup skip ZIL_s46: pybit не установлен. pip install pybit
Warmup skip ZIL_s49: pybit не установлен. pip install pybit
Warmup skip ZK_s0: pybit не установлен. pip install pybit
Warmup skip ZK_s1: pybit не установлен. pip install pybit
Warmup skip ZK_s2: pybit не установлен. pip i

Warmup skip ZKP_s0: pybit не установлен. pip install pybit
Warmup skip ZKP_s1: pybit не установлен. pip install pybit
Warmup skip ZKP_s2: pybit не установлен. pip install pybit
Warmup skip ZKP_s3: pybit не установлен. pip install pybit
Warmup skip ZKP_s9: pybit не установлен. pip install pybit
Warmup skip ZKP_s11: pybit не установлен. pip install pybit
Warmup skip ZKP_s14: pybit не установлен. pip install pybit
Warmup skip ZKP_s15: pybit не установлен. pip install pybit
Warmup skip ZKP_s16: pybit не установлен. pip install pybit
Warmup skip ZKP_s17: pybit не установлен. pip install pybit
Warmup skip ZKP_s18: pybit не установлен. pip install pybit
Warmup skip ZKP_s20: pybit не установлен. pip install pybit
Warmup skip ZKP_s27: pybit не установлен. pip install pybit
Warmup skip ZKP_s34: pybit не установлен. pip install pybit
Warmup skip ZKP_s35: pybit не установлен. pip install pybit
Warmup skip ZKP_s36: pybit не установлен. pip install pybit
Warmup skip ZKP_s37: pybit не установлен. pip

Warmup skip ZRO_s0: pybit не установлен. pip install pybit
Warmup skip ZRO_s1: pybit не установлен. pip install pybit
Warmup skip ZRO_s2: pybit не установлен. pip install pybit
Warmup skip ZRO_s3: pybit не установлен. pip install pybit
Warmup skip ZRO_s4: pybit не установлен. pip install pybit
Добавлено warmup-строк (Bybit): 0


Warmup применён: size=60. После удаления warmup строк: 365,375
Фичей: 25


,rd_mom_1,rd_mom_5,rd_mom_10,rd_acceleration,rd_zscore_30,rd_regime,rd_regime_transition
0,NaN,NaN,NaN,NaN,0.000000,-1,0
1,-0.032185,NaN,NaN,NaN,-0.707107,-1,0
2,-0.001196,NaN,NaN,0.030989,-0.608650,-1,0
3,-0.006897,NaN,NaN,-0.005701,-0.767726,-1,0
4,-0.023221,NaN,NaN,-0.016324,-1.302774,-1,0


## 5. Сохранение prepared данных

In [5]:
out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'prepared_with_rd_regime.parquet')
df_save = df.copy()
df_save['symbol'] = df_save['symbol'].astype(str)
df_save.to_parquet(out_path, index=False, compression='snappy')
print(f'Сохранено: {out_path} ({len(df):,} строк)')

Сохранено: c:\project\trading_bot_2Engine\outputs\prepared_with_rd_regime.parquet (365,375 строк)
